In [1]:
# Import splish
import pandas as pd
import numpy as np
import re
import os
from nltk.inference.prover9 import *

os.environ["PROVER9"] = "/home/flopezp/Prover9/bin/prover9"

In [2]:
folio_full_val = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_validation.jsonl', lines = True)
folio_full_test = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_test.jsonl', lines = True)
trying_splish = pd.read_csv('/home/flopezp/Kurosagol/Ongoing/baseline_datasets/test/filtered/translation/TRANS_DeepSeek-R1-0528-Qwen3-8B.csv')
trying_splish = trying_splish.drop(columns = ["Unnamed: 0"])
trying_splish.head()

,Translation
0,['∀x (ProfessionalSoccerPlayer(x) → ¬Professio...
1,['∀x (ProfessionalSoccerPlayer(x) → ¬Professio...
2,['∀x (ProfessionalSoccerPlayer(x) → ¬Professio...
3,"['∀x (JoinSG(x) → ImproveComm(x)) ', '∀x (Stay..."
4,"['∀x (JoinSG(x) → ImproveComm(x)) ', '∀x (Stay..."


In [7]:
def prove(argument):
    goal, assumptions = argument
    g = Expression.fromstring(goal)
    alist = [Expression.fromstring(a) for a in assumptions]
    p = Prover9Command(g, assumptions=alist).prove()
    return p

# FOL to Prover9 Syntax

def switch_quantifiers(text, cuantifier):
    """
        Elimina todos los cuantificadores y los traduce a sintaxis de Prover9. Sin importar la variable ni la cantidad de apariciones. Qué pedo soy una verga para esto.

        text = str ;  Texto a modificar.
        cuantifier = str ('forall', 'exists') ; Cuantificador a modificar.
    """
    if cuantifier == 'forall':
        regex = '∀[A-z]'
        cuant = 'all '
    else:
        regex = '∃[A-z]'
        cuant = 'exists '

    owo = re.finditer(regex, text)
    aux_list = list(owo)
    if len(aux_list) == 0:
        return text

    # Redefinimos el iterador porque hacer lista de un iterador lo consume. CHINGA TU MADRE PYTHON. VETE A LA BURGER.    
    owo = re.finditer(regex, text)
    temporal_str = ''

    for _ in owo:
        if temporal_str == '':
            temporal_str = text[0:_.start()] + cuant + _.group()[-1] + ' ' + text[_.end():]
        else:
            value = re.search(regex, temporal_str)
            temporal_str = temporal_str[0:(value.start())] + cuant + value.group()[-1] + ' ' + temporal_str[value.end():]
    
    return temporal_str



def fol_to_prover9(value):
    """
        Modifica los símbolos lógicos normales y los cambia por los valores adecuados para Prover9.*
    """
    temp = value.lower()
    temp = switch_quantifiers(temp, 'forall')
    temp = switch_quantifiers(temp, 'exists')
    temp = re.sub('-', '', temp)
    temp = re.sub('¬', ' -', temp) 
    temp = re.sub('→|→', '->', temp)
    temp = re.sub('∧', '&', temp)
    temp = re.sub('∨', '|', temp)
    temp = re.sub('↔', '<->', temp)
    temp = re.sub('≠', '!=', temp)
    temp = re.sub(r'\'', '', temp)
    #temp = re.sub(r'[\'\"]', '', temp)
    #temp = re.sub(r'\[', '(', temp)
    #temp = re.sub(r'\]', ')', temp)
    temp = re.sub(r'\[', '', temp)
    temp = re.sub(r'\]', '', temp)
    temp = re.sub('∴', '', temp)
    temp = re.sub(r'(\. \()', '.(', temp)
    temp = re.sub(r'\.{2}', '.', temp)
    temp = re.sub(r'\)\.', ')', temp)
    temp = re.sub(r'([^a-z]\.\()', '(', temp)
    temp = re.sub(r'\.', ' ', temp)
    temp = re.sub('(  )+', ' ', temp)
    #temp = temp + '.'
    return temp


def dash_predicates(text):
    """
        Cambia los predicados de la forma "texto-texto-texto(x)" -> "textotextotexto(x)"

        text = str ; el hilo a modificar.
    """
    all_values = len(re.findall(r'[a-z0-9]+(\-[a-z0-9]+\-{0,})+[a-z0-9]+', text))

    if all_values == 0:
        return text
    
    new_text = text
    for i in range(all_values):
        current_regex = re.search(r'[a-z0-9]+(\-[a-z0-9]+\-{0,})+[a-z0-9]+', new_text)
        split = current_regex.group().split()
        aux_text = ''
        for elem in split:
            aux_text = aux_text + elem
        new_text = new_text[:current_regex.start()] + aux_text + new_text[current_regex.end():]

    return new_text


def elim_spaces(text):
    """
        Elimina los espacios entre variables: lionel messi -> lionelmessi

        text = str; El texto a modificar.

        OBS: Este formato de funciones (Encontrar cantidades y luego iterar sobre las cantidades) me gusta bastante.
    """
    total_iters = len(list(re.finditer(r'([A-z]+ )+([A-z]{2,})', text)))
    if total_iters == 0:
        return text

    new_text = text
    for i in range(total_iters):
        current_regex = re.search(r'([A-z]+ )+([A-z]{2,})', new_text)
        split = current_regex.group().split()
        aux_text = ''
        for elem in split:
            aux_text = aux_text + elem
        new_text = new_text[:current_regex.start()] + aux_text + new_text[current_regex.end():]

    return new_text


# El XOR me tiene hasta los huevos cabrón te lo juro.
def xor_bonito(expression):
    """
        Elimina el símbolo de XOR, y lo reescribe en la fórmula (A OR B) AND NOT(A AND B)
    """
    individual_values = re.findall(r'([A-z|_|0-9]{2,}|¬)', expression)
    a = individual_values[0] + '(x)'
    b = individual_values[1] + '(x)'
    a_or_b = '(' + a + ' | ' + b + ')'
    not_a_and_b = ' -(' + a + ' & ' + b +')'
    xor = a_or_b + ' & ' + not_a_and_b
    return xor

def xor_bonito_extreme(expression):
    """
        Elimina los XOR de fórmulas compuestas.
    """
    aux2 = re.search(r'([a-z]+\([a-z, ]+\)) ⊕ ([a-z]+\([a-z, ]+\))', expression)
    split = aux2.group().split('⊕')
    a_f = split[0]
    b_f = split[-1]
    a_or_b = '(' + a_f + ' | ' + b_f + ')'
    not_a_and_b = ' -(' + a_f + ' & ' + b_f +')'
    xor = a_or_b + ' & ' + not_a_and_b
    return xor

def rewrite_xor(re_search, element, comp):
    """
        Genera una nueva expresión a partir del xor bonito. 

        re_search = re.search(regex, str)
        element = str ; same str as above
        comp = bool ; True iff predicates have multiple variables.
    """
    start = re_search.start()
    end = re_search.end()

    xor_substr = element[start:end]
    if comp:
        xor_chido = xor_bonito_extreme(xor_substr)
    else:
        xor_chido = xor_bonito(xor_substr)

    nuevo = element[:start] + xor_chido + element[end:]
    return nuevo


def clean(value, FOLIO):
    """
        Procesa una respuesta individual de FOLIO/GPT_TRANS/QWEN_TRANS para que se pase al formato de Prover9.
    """
    if FOLIO:
        clean_premises_aux = value.split('\n')
    else:
        clean_premises_aux = value.split('\',')
    clean_premises = []
    for _ in clean_premises_aux:
        if _ != '':
            clean_premises.append(_)

    if "Premises" in clean_premises[0]:
        del clean_premises[0]

    #print(clean_premises[0])

    for _ in clean_premises:
        clean_premises[clean_premises.index(_)] = re.sub(r'(:::)+([ A-z.]+)', '', _)

    for _ in clean_premises:
        clean_premises[clean_premises.index(_)] = re.sub(r'[0-9]\.', '', _)
    
    for instance in clean_premises:
        clean_premises[clean_premises.index(instance)] = fol_to_prover9(instance)

    #print(clean_premises[0])
    for instance in clean_premises:
        clean_premises[clean_premises.index(instance)] = elim_spaces(instance)

    try:
        # Filtro XOR sencillo
        for element in clean_premises:
            xor_count = len(re.findall('⊕', element))
            if xor_count > 0:
                value = element
                while xor_count > 0:
                    aux = re.search(r'(-{0,1}[a-z]+\([a-z, ]+\)) ⊕ (-{0,1}[a-z]+\([a-z, ]+\))', element)
                    element = rewrite_xor(aux, element, False)
                    xor_count = len(re.findall('⊕', element))
                clean_premises[clean_premises.index(value)] = element

        # Filtro XOR multipremisa
        for element in clean_premises:
            xor_count = len(re.findall('⊕', element))
            if xor_count > 0:
                value = element
                while xor_count > 0:
                    aux = re.search(r'(-{0,1}[a-z]+\([a-z, ]+\)) ⊕ (-{0,1}[a-z]+\([a-z, ]+\))', element)
                    element = rewrite_xor(aux, element, True)
                    xor_count = len(re.findall('⊕', element))
                clean_premises[clean_premises.index(value)] = element

    except:
        print("Hubo un Xor malo")    

    return clean_premises



# ========================================
# ========================================
# ========================================

def check_arg_validity(dataset, index, validation):
    """
        Dadas premisas en lenguaje lógico, verifica que la conclusión correspondiente sea inferible.
    """
    if validation:
        folio_full = folio_full_val
    else:
        folio_full= folio_full_test
    llm_valid, llm_invalid, fol_valid, fol_invalid = 0, 0, 0, 0
    #if llm == 'qwen':
    #    llm_ds = full_qwen
    #else:
    #    llm_ds = full_gpt
    #llm_value = llm_ds['Translation'.format(llm)][index]
    llm_value = dataset['Translation'][index]
    folio_value = folio_full['premises-FOL'][index]

    true_validity = folio_full['label'][index]
    print("Logical Validity: {}".format(true_validity))

    clean_llm = clean(llm_value, False)
    print('Clean LLM: ', clean_llm)
    clean_folio = clean(folio_value, True)
    print('Clean FOLIO: ', clean_folio)
    clean_conc = clean(folio_full['conclusion-FOL'][index], True) #OJO AQUÍ / EYE HERE
    print('Clean Conc: ', clean_conc)
    
    args_llm = (
        clean_conc[0],
        clean_llm
    )

    args_folio = (
        clean_conc[0],
        clean_folio
    )
    
    # Lo que hacemos aquí es evaluar una conclusión contra las premisas de un LLM o del conjunto de datos.
    # OBS: A veces la premisa NO debe de ser deducible. 
    # ¿Cómo vamos a cuantificar este show?
    # Casos:
    # Premisas: TRUE, UNCERTAIN, FALSE.
    # Posibles resultados: TRUE, FALSE. ¿Cómo lidiamos con Uncertain? -> En estos casos Prover9 te regresa False. 
    
    #print("LLM ({}):".format())
    
    try:
        print("Ejecutable. Valor: {}".format(prove(args_llm)))
        llm_valid += 1
    except Exception as e:
        print(e)
        print(type(e))
        llm_invalid += 1
        
    print("FOLIO:")
    try:
        print("Ejecutable. Valor: {}".format(prove(args_folio)))
        fol_valid += 1
    except Exception as e:
        print(e)
        print(type(e))
        fol_invalid += 1
    print("======================")

    return llm_valid, llm_invalid, fol_valid, fol_invalid
    

In [8]:
a = clean(folio_full_test['premises-FOL'][2], True)
b = clean(folio_full_test['conclusion-FOL'][2], True)

args_folio = (
    b[0],
    a
)
prove(args_folio)

True

# Ahora solo tenemos que hacer las generaciones de cada modelo PARA LOS VALORES VERDADEROS. Una vez que tengamos eso, es cuestión de ejecutar esto.

In [9]:
for i in range(len(folio_full_test['premises-FOL'])):
    a = clean(folio_full_test['premises-FOL'][i], True)
    b = clean(folio_full_test['conclusion-FOL'][i], True)
    for val in a:
        print(val)
    print(b[0])
    print(folio_full_test['label'][i])
    print('-------')

all x ((professional(x) & soccerplayer(x)) -> -play(x, professionalbasketball))
all x ((professional(x) & defender(x)) -> (professional(x) & soccerplayer(x)))
all x ((professional(x) & centerback(x)) -> (professional(x) & defender(x)))
exists x (athlete(x) & professional(x) & centerback(x))
play(stephencurry, professionalbasketball)
athlete(stephencurry)
Uncertain
-------
all x ((professional(x) & soccerplayer(x)) -> -play(x, professionalbasketball))
all x ((professional(x) & defender(x)) -> (professional(x) & soccerplayer(x)))
all x ((professional(x) & centerback(x)) -> (professional(x) & defender(x)))
exists x (athlete(x) & professional(x) & centerback(x))
play(stephencurry, professionalbasketball)
athlete(stephencurry) & professional(stephencurry) & centerback(stephencurry)
False
-------
all x ((professional(x) & soccerplayer(x)) -> -play(x, professionalbasketball))
all x ((professional(x) & defender(x)) -> (professional(x) & soccerplayer(x)))
all x ((professional(x) & centerback(x)

In [10]:
_llm_valid, _llm_invalid, _fol_valid, _fol_invalid = 0,0,0,0
for i in range(len(trying_splish['Translation'])):
    print(i)
    #try:
    a, b, c, d = check_arg_validity(trying_splish, i, False)
    _llm_valid += a
    _llm_invalid += b
    _fol_valid += c
    _fol_invalid += d
    #except:
    #    print("oops")

print("Premisas de LLM válidas: {}".format(_llm_valid))
print("Premisas de LLM inválidas: {}".format(_llm_invalid))
print("Premisas de FOLIO válidas: {}".format(_fol_valid))
print("Premisas de FOLIO inválidas: {}".format(_fol_invalid))

0
Logical Validity: Uncertain
Clean LLM:  ['all x (professionalsoccerplayer(x) -> -professionalbasketballplayer(x)) ', ' all x (professionalsoccerdefender(x) -> professionalsoccerplayer(x)) ', ' all x (professionalcenterback(x) -> professionalsoccerdefender(x)) ', ' exists x (athlete(x) & professionalcenterback(x)) ', ' professionalbasketballplayer(stephencurry) ']
Clean FOLIO:  ['all x ((professional(x) & soccerplayer(x)) -> -play(x, professionalbasketball))', 'all x ((professional(x) & defender(x)) -> (professional(x) & soccerplayer(x)))', 'all x ((professional(x) & centerback(x)) -> (professional(x) & defender(x)))', 'exists x (athlete(x) & professional(x) & centerback(x))', 'play(stephencurry, professionalbasketball)']
Clean Conc:  ['athlete(stephencurry)']
Ejecutable. Valor: False
FOLIO:
Ejecutable. Valor: False
1
Logical Validity: False
Clean LLM:  ['all x (professionalsoccerplayer(x) -> -professionalbasketballplayer(x)) ', ' all x (professionalsoccerdefender(x) -> professionalso